# Etapa 3. Modelo hibrido

### 1. Inicialización

In [1]:
import pandas as pd
import numpy as np

from nltk.corpus import stopwords as nltk_stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import pickle

### 2. Carga de datos

In [2]:
data = pd.read_csv('datasets/steam_games_preprocessed.csv')

In [3]:
tf_idf = pickle.load(open('models/tfidf_vectorizer.pkl', 'rb'))
tf_idf_desc_matrix = pickle.load(open('models/tfidf_matrix.pkl', 'rb'))

### 3. Matriz TF-IDF y de similitud para generos

In [4]:

stop_words = list(nltk_stopwords.words('english'))

count_tf_idf_genres = TfidfVectorizer(stop_words=stop_words)
tf_idf_genres_matrix = count_tf_idf_genres.fit_transform(data['genres'])

sim_genres_matrix = cosine_similarity(tf_idf_genres_matrix)

### 4. Matriz TF-IDF y similitud para etiquetas

In [5]:

count_tf_idf_tags = TfidfVectorizer(stop_words=stop_words)
tf_idf_tags_matrix = count_tf_idf_tags.fit_transform(data['steamspy_tags'])

sim_tags_matrix = cosine_similarity(tf_idf_tags_matrix)

### 5. Matríz hibrida
Con el fin de obtener un modelo que recomiende de manera más robusta, contruiremos una matriz de similutud que se base no solo en las descripciones si no tambien en las etiquetas y en los géneros a los que pertenece cada videojuego. Elegiremos un peso de 60% para las descripciones ya que nos interesa evaluar nuestro modelo de aprendizaje de textos, 25% a las etiquetas y 15% a los generos para apoyar nuestras clasificaciones en ellos tambien.

In [6]:
#Carga de matriz a partir de las descripciones
sim_description_matrix = pickle.load(open('models/similarity_matrix.pkl', 'rb'))

In [7]:
#Matriz hibrida: para esta daremos un valor de 60% a las descripciones, 25% a las etiquetas y 15% a los géneros 
sim_hybrid_matrix = (
    (0.6 * sim_description_matrix) +
    (0.25 * sim_tags_matrix) +
    (0.15 * sim_genres_matrix)
)

In [8]:
#guardemos nuestra matriz
with open('models/similarity_hybrid.pkl', 'wb') as f:
    pickle.dump(sim_hybrid_matrix, f)

### 6. Función de recomendación
Crearemos la funcion que recomienda los videojuegos más parecidos al indicado ahora basandonos en nuestra nueva matríz de similitud.

In [9]:
def recommend_hybrid(game_name, data, similarity_matrix, top_n=5):
    if game_name not in data['name'].values:
        raise ValueError('No encontrado en la base de datos')

    game_index = data.index[data['name'] == game_name][0]

    sim_scores = list(enumerate(similarity_matrix[game_index]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]

    recommended_games = data.iloc[[index for index, _ in sim_scores]][['appid', 'name', 'genres', 'steamspy_tags']]
    
    return recommended_games
    

In [10]:
#Probemos la función
print(data[data['name'] == 'Counter-Strike'][['name', 'genres', 'steamspy_tags']])
recommend_hybrid('Counter-Strike', data, sim_hybrid_matrix)

             name  genres             steamspy_tags
0  Counter-Strike  Action  Action, FPS, Multiplayer


,appid,name,genres,steamspy_tags
1,20,Team Fortress Classic,Action,"Action, FPS, Multiplayer"
203,7940,Call of Duty® 4: Modern Warfare®,Action,"FPS, Action, Multiplayer"
10,240,Counter-Strike: Source,Action,"Action, FPS, Multiplayer"
258,10180,Call of Duty®: Modern Warfare® 2,Action,"Action, FPS, Multiplayer"
5,60,Ricochet,Action,"Action, FPS, Multiplayer"


Vemos que definitivamente las recomendaciones cambian ahora que hemos agregado más criterios de recomendación, obtenemos recomendaciones mucho más robustas con generos y etiquetas más parecidas.

In [11]:
#Guardemos la función
with open('models/recommend_hybrid_function.pkl', 'wb') as f:
    pickle.dump(recommend_hybrid, f)

### 7. Conclusiones

Hemos logrado crear nuestra matriz de similitud hibrida compuesta por:

1. Similitud por descripción (TF-IDF)
2. Similitud por tags
3. Similitud por géneros

Además creamos nuestra función de similitud que nos permitirá encontrar los juegos más parecidos al indicado basado no solo en la descripción sinó que además se apoyará tambien en los valores de genero y etiquetas en nuestros datos para reforzar sus recomendaciones.